In [1]:
import numpy as np
from sklearn.datasets import fetch_olivetti_faces
from sklearn.tree import DecisionTreeClassifier
import logging

# Configure logging to overwrite "_.log" every run (mode="w")
logging.basicConfig(
    level=logging.INFO,
    format="%(message)s",
    handlers=[
        logging.FileHandler("q1_a.log", mode="w"),  # "w" overwrites, "a" appends
        logging.StreamHandler()
    ],
    force=True  # Resets any existing logging handlers
)
logger = logging.getLogger(__name__)

faces = fetch_olivetti_faces(shuffle=False)
X = faces.images
y = faces.target

height = X.shape[1]
width = X.shape[2]

split_data = np.load("olivetti_split_indices.npz")

train_idx = split_data["train"]
val_idx = split_data["val"]
test_idx = split_data["test"]

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
X_test, y_test = X[test_idx], y[test_idx]

X_test_corrupted = X_test.copy()
X_test_corrupted[:, 24:40, 24:40] = 0.0

logger.info(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
logger.info(f"X_val:   {X_val.shape}, y_val:   {y_val.shape}")
logger.info(f"X_test:  {X_test.shape}, y_test:  {y_test.shape}")
logger.info(f"X_test_corrupted:  {X_test_corrupted.shape}, y_test:  {y_test.shape}")




X_train = X_train.reshape(-1, height * width)
X_val = X_val.reshape(-1, height * width)
X_test = X_test.reshape(-1, height * width)
X_test_corrupted = X_test_corrupted.reshape(-1, height * width)

seed = 260007
sc = 0
max_depths = [2, 4, 8, 16, None]
min_samples_leafs = [1, 2, 4]

best_val_acc = -1.0
best_params = None
best_model = None

results = []

for depth in max_depths:
    for leaf in min_samples_leafs:
        clf = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=leaf, random_state=seed + sc)
        clf.fit(X_train, y_train)
        
        train_acc = clf.score(X_train, y_train)
        val_acc = clf.score(X_val, y_val)
        
        results.append((depth, leaf, train_acc, val_acc, clf))
        logger.info(f"max_depth:{str(depth)} | min_samples_leaf:{leaf} | Train Acc:{train_acc:.4f} | Val Acc:{val_acc:.4f}")
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_params = (depth, leaf)
            best_model = clf

        sc = sc + 1

logger.info("-" * 55)
logger.info(f"Selected Best Parameters (Frozen): max_depth={best_params[0]}, min_samples_leaf={best_params[1]}")

tree = best_model.tree_

is_leaf = (tree.children_left == -1) & (tree.children_right == -1)
num_leaves = np.sum(is_leaf)

leaf_sample_counts = tree.n_node_samples[is_leaf]
mean_samples_per_leaf = np.mean(leaf_sample_counts)

single_sample_leaves_count = np.sum(leaf_sample_counts == 1)
fraction_single_sample_leaves = single_sample_leaves_count / num_leaves

final_train_acc = best_model.score(X_train, y_train)
final_val_acc = best_model.score(X_val, y_val)
final_clean_test_acc = best_model.score(X_test, y_test)
final_corrupted_test_acc = best_model.score(X_test_corrupted, y_test)

logger.info("\n=== Single Tree Baseline Summary Report ===")
logger.info(f"Best Hyperparameters          : max_depth={best_params[0]}, min_samples_leaf={best_params[1]}")
logger.info(f"Train Accuracy               : {final_train_acc:.4f}")
logger.info(f"Validation Accuracy          : {final_val_acc:.4f}")
logger.info(f"Clean Test Accuracy          : {final_clean_test_acc:.4f}")
logger.info(f"Corrupted Test Accuracy      : {final_corrupted_test_acc:.4f}")
logger.info(f"Number of Leaves             : {num_leaves}")
logger.info(f"Mean Samples per Leaf        : {mean_samples_per_leaf:.2f}")
logger.info(f"Fraction of Single-Sample Leaves: {fraction_single_sample_leaves:.4f} ({fraction_single_sample_leaves*100:.2f}%)")

X_train: (240, 64, 64), y_train: (240,)
X_val:   (80, 64, 64), y_val:   (80,)
X_test:  (80, 64, 64), y_test:  (80,)
X_test_corrupted:  (80, 64, 64), y_test:  (80,)
max_depth:2 | min_samples_leaf:1 | Train Acc:0.0750 | Val Acc:0.0750
max_depth:2 | min_samples_leaf:2 | Train Acc:0.0750 | Val Acc:0.0625
max_depth:2 | min_samples_leaf:4 | Train Acc:0.0750 | Val Acc:0.0750
max_depth:4 | min_samples_leaf:1 | Train Acc:0.1250 | Val Acc:0.1125
max_depth:4 | min_samples_leaf:2 | Train Acc:0.1250 | Val Acc:0.1000
max_depth:4 | min_samples_leaf:4 | Train Acc:0.1250 | Val Acc:0.1250
max_depth:8 | min_samples_leaf:1 | Train Acc:0.2250 | Val Acc:0.1875
max_depth:8 | min_samples_leaf:2 | Train Acc:0.2250 | Val Acc:0.1750
max_depth:8 | min_samples_leaf:4 | Train Acc:0.2250 | Val Acc:0.1875
max_depth:16 | min_samples_leaf:1 | Train Acc:0.4250 | Val Acc:0.3500
max_depth:16 | min_samples_leaf:2 | Train Acc:0.4250 | Val Acc:0.3375
max_depth:16 | min_samples_leaf:4 | Train Acc:0.4250 | Val Acc:0.3250
max_d